In [11]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent

sys.path.append(str(PROJECT_ROOT))

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [12]:
from src.pipelines.movie_pipeline import MoviePipeline

pipeline = MoviePipeline()

datasets = pipeline.load_datasets()

# movies = datasets["movies"]
# ratings = datasets["ratings"]
# links = datasets["links"]
# tags = datasets["tags"]

In [13]:
print(datasets["movies"].shape)
print(datasets["ratings"].shape)
print(datasets["links"].shape)
print(datasets["tags"].shape)

(9742, 3)
(100836, 4)
(9742, 3)
(3683, 4)


In [14]:
rating_summary = pipeline.aggregate_ratings(datasets["ratings"])

In [15]:
rating_summary.head()

,movieId,average_rating,rating_count
0,1,3.92,215
1,2,3.43,110
2,3,3.26,52
3,4,2.36,7
4,5,3.07,49


In [16]:
rating_summary.shape

(9724, 3)

In [17]:
rating_summary.describe()

,movieId,average_rating,rating_count
count,9724.000000,9724.000000,9724.000000
mean,42245.024373,3.262465,10.369807
std,52191.137320,0.869896,22.401005
min,1.000000,0.500000,1.000000
25%,3245.500000,2.800000,1.000000
50%,7300.000000,3.420000,3.000000
75%,76739.250000,3.910000,9.000000
max,193609.000000,5.000000,329.000000


In [18]:
tag_summary = pipeline.aggregate_tags(datasets["tags"])

In [19]:
tag_summary.head()

,movieId,tags
0,1,pixar pixar fun
1,2,fantasy magic board game Robin Williams game
2,3,moldy old
3,5,pregnancy remake
4,7,remake


In [20]:
tag_summary.shape

(1572, 2)

In [22]:
movie_features = pipeline.merge_datasets(datasets["movies"],rating_summary,datasets["links"],tag_summary)

In [23]:
movie_features.head()

,movieId,title,genres,average_rating,rating_count,tags,imdbId,tmdbId
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,3.92,215.0,pixar pixar fun,114709,862.0
1,2,Jumanji (1995),Adventure|Children|Fantasy,3.43,110.0,fantasy magic board game Robin Williams game,113497,8844.0
2,3,Grumpier Old Men (1995),Comedy|Romance,3.26,52.0,moldy old,113228,15602.0
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,2.36,7.0,NaN,114885,31357.0
4,5,Father of the Bride Part II (1995),Comedy,3.07,49.0,pregnancy remake,113041,11862.0


In [24]:
movie_features.shape

(9742, 8)

In [25]:
movie_features.describe()

,movieId,average_rating,rating_count,imdbId,tmdbId
count,9742.000000,9724.000000,9724.000000,9.742000e+03,9734.000000
mean,42200.353623,3.262465,10.369807,6.771839e+05,55162.123793
std,52160.494854,0.869896,22.401005,1.107228e+06,93653.481487
min,1.000000,0.500000,1.000000,4.170000e+02,2.000000
25%,3248.250000,2.800000,1.000000,9.518075e+04,9665.500000
50%,7300.000000,3.420000,3.000000,1.672605e+05,16529.000000
75%,76232.000000,3.910000,9.000000,8.055685e+05,44205.750000
max,193609.000000,5.000000,329.000000,8.391976e+06,525662.000000


In [26]:
movie_features.info()

<class 'pandas.DataFrame'>
RangeIndex: 9742 entries, 0 to 9741
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   movieId         9742 non-null   int64  
 1   title           9742 non-null   str    
 2   genres          9742 non-null   str    
 3   average_rating  9724 non-null   float64
 4   rating_count    9724 non-null   float64
 5   tags            1572 non-null   str    
 6   imdbId          9742 non-null   int64  
 7   tmdbId          9734 non-null   float64
dtypes: float64(3), int64(2), str(3)
memory usage: 1.0 MB


In [27]:
movie_features.isnull().sum()

movieId              0
title                0
genres               0
average_rating      18
rating_count        18
tags              8170
imdbId               0
tmdbId               8
dtype: int64

## Observations

- Total no. of movies : 9742
- Highest AVG Rating : 5
- lowest AVG Rating : 0.5
- Maximum rating count : 329
- Minimum rating count : 1
- Average number of ratings per movie : 3.26

- "average_rating" and "rating_count" both have 18 unrated movies !

- "tags" column contain 8170 untagged moives! which means only 1572 movies have been tagged by users

## Engineering Insights

1. The MovieLens dataset is highly sparse.
2. Average rating alone is not a reliable popularity measure.
3. Most movies have very few ratings.
4. Only a small percentage of movies contain user-generated tags.
5. Multiple datasets will be required to build a high-quality content-based recommender.